# Introduction to Computer Vision: MNIST Handwritten Digit Classification

**Companion video case study (CV / image classification)**

MNIST is the classic "Hello World" of computer vision: 70,000 grayscale handwritten
digit images of size 28×28. The task is to assign each image to one of the **10 classes
0–9**. This notebook walks you through the complete pipeline from scratch:

1. **Load and visualize** the data to build the intuition that "an image is just a matrix of numbers";
2. **Preprocess**: normalize, split into train/test sets, and batch with `DataLoader`;
3. Train and compare two models — first flatten the pixels and feed them to an
   **MLP (multilayer perceptron)**, then use a **CNN (convolutional neural network)**
   that actually "looks at" the image;
4. **Evaluate**: accuracy, confusion matrix, and inspect which samples the model gets wrong.

> It's recommended to **run the cells from top to bottom**. CPU is enough throughout
> (training the CNN for 1 epoch takes about 1–2 minutes).


## 1. Environment and dependencies

This case study needs `torch` and `torchvision` (used to automatically download MNIST).
If they aren't installed yet:

```bash
pip install torch torchvision matplotlib
```

The first run downloads MNIST (about 11 MB) into the `./data/` directory; afterwards it
reads directly from the cache.


In [ ]:
import time

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

import torchvision
from torchvision import datasets, transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(0)
np.random.seed(0)

print('torch       :', torch.__version__)
print('torchvision :', torchvision.__version__)
print('device      :', device)

## 2. Load the data

`torchvision.datasets.MNIST` automatically downloads and parses the dataset. We use
`transforms` to convert each PIL image into a tensor of shape `(1, 28, 28)` (1 grayscale
channel) and **standardize** it: subtract the global mean `0.1307` and divide by the
standard deviation `0.3081` (these two numbers are statistics of the MNIST training set
and have become a convention).

Standardization keeps pixel values roughly centered around 0 with unit variance, which
makes training more stable and convergence faster.


In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),                       # PIL -> Tensor, pixels scaled to [0, 1]
    transforms.Normalize((0.1307,), (0.3081,)),  # standardize: (x - mean) / std
])

train_set = datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_set  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

print('Training samples:', len(train_set))
print('Test samples    :', len(test_set))

img, label = train_set[0]
print('Single image tensor shape:', tuple(img.shape), '  label:', label)
print('Pixel value range: [%.2f, %.2f]' % (img.min().item(), img.max().item()))

## 3. Visualize samples

"An image is just a matrix of numbers." Let's first plot a batch of images to see what
they look like, then print the raw pixel values of one of them. This builds the intuition
that the model doesn't see a "picture" — it sees a 28×28 matrix of floating-point numbers.


In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(12, 3.2))
for ax, idx in zip(axes.ravel(), range(16)):
    img, label = train_set[idx]
    ax.imshow(img.squeeze(), cmap='gray')   # squeeze: (1,28,28) -> (28,28)
    ax.set_title(str(label))
    ax.axis('off')
fig.suptitle('MNIST samples (title is the true label)')
plt.tight_layout()
plt.show()

In [ ]:
# Print the pixel matrix of one image: brighter pixels (strokes) have larger values
img, label = train_set[0]
arr = img.squeeze().numpy()
print('This is the digit:', label, '   matrix shape:', arr.shape)
plt.figure(figsize=(6, 6))
plt.imshow(arr, cmap='gray')
for i in range(28):
    for j in range(28):
        v = arr[i, j]
        if v > -1.0:  # only annotate the stroke region to avoid clutter
            plt.text(j, i, '%.0f' % (v * 10), ha='center', va='center',
                     color='red', fontsize=5)
plt.title('Pixel values of digit %d (×10, rounded)' % label)
plt.axis('off')
plt.show()

## 4. A quick look at the class distribution

In a classification task, first check whether the classes are roughly balanced — if some
digits have very few samples, accuracy can become biased. The 10 digits in MNIST are
fairly evenly distributed.


In [ ]:
labels = train_set.targets.numpy()
counts = np.bincount(labels, minlength=10)
plt.figure(figsize=(7, 3))
plt.bar(range(10), counts)
plt.xticks(range(10))
plt.xlabel('Digit class'); plt.ylabel('Number of training samples')
plt.title('Samples per class (roughly balanced)')
plt.show()
print('Samples per class:', counts)

## 5. Batching: DataLoader

During training we don't feed data one image at a time but in **batches**. `DataLoader`
handles shuffling and fetching data batch by batch. Here `batch_size=128`, the training
set is shuffled and the test set is not.


In [ ]:
batch_size = 128
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_set,  batch_size=batch_size, shuffle=False)

xb, yb = next(iter(train_loader))
print('Image tensor of one batch:', tuple(xb.shape))  # (128, 1, 28, 28)
print('Label tensor of one batch:', tuple(yb.shape))  # (128,)

## 6. Reusable training and evaluation functions

To compare the two models fairly, we write two reusable functions:
- `train_one_epoch`: run through the entire training set once — forward pass, compute loss,
  backpropagate, and update parameters;
- `evaluate`: compute the **average loss** and **accuracy** on the test set (without
  updating parameters).

For classification we use **cross-entropy loss** (`nn.CrossEntropyLoss`), which already
includes softmax internally, so the model's last layer just outputs 10 raw scores (logits).


In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        total_loss += criterion(logits, yb).item() * xb.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == yb).sum().item()
    n = len(loader.dataset)
    return total_loss / n, correct / n


def fit(model, epochs, lr=1e-3):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    for ep in range(1, epochs + 1):
        t0 = time.time()
        tr_loss = train_one_epoch(model, train_loader, optimizer, criterion)
        te_loss, te_acc = evaluate(model, test_loader, criterion)
        history.append((tr_loss, te_loss, te_acc))
        print('epoch %d/%d | train_loss %.4f | test_loss %.4f | test_acc %.4f | %.1fs'
              % (ep, epochs, tr_loss, te_loss, te_acc, time.time() - t0))
    return history

## 7. Model A: Multilayer Perceptron (MLP)

The most naive approach: **flatten** the 28×28 image into a 784-dimensional vector and
feed it into a few fully connected layers. This completely ignores the 2D structure
("neighboring pixels are more correlated"), but as a baseline it already reaches about
**97–98%** accuracy.

Architecture: `784 → 256 → 128 → 10`, with ReLU activations in between and a bit of
Dropout to prevent overfitting.


In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),            # (B,1,28,28) -> (B,784)
            nn.Linear(28 * 28, 256), nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),     nn.ReLU(),
            nn.Linear(128, 10),      # output 10 logits
        )

    def forward(self, x):
        return self.net(x)


mlp = MLP()
n_params = sum(p.numel() for p in mlp.parameters())
print('MLP parameter count:', n_params)
print(mlp)

In [ ]:
print('===== Training MLP =====')
hist_mlp = fit(mlp, epochs=3)

## 8. Model B: Convolutional Neural Network (CNN)

A CNN slides **convolution kernels** over the image and automatically learns local
features like "edges, strokes, corners," which naturally fits the 2D structure of images.
Here we use a small LeNet-style network:

```
Conv(1→16) → ReLU → MaxPool      # 28x28 -> 14x14
Conv(16→32) → ReLU → MaxPool     # 14x14 -> 7x7
Flatten → Linear(32*7*7 → 128) → ReLU → Dropout → Linear(128 → 10)
```

It has even fewer parameters than the MLP, but because it exploits spatial structure,
accuracy typically reaches around **99%**.


In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),                              # 28 -> 14
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),                              # 14 -> 7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128), nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


cnn = CNN()
print('CNN parameter count:', sum(p.numel() for p in cnn.parameters()))
print(cnn)

In [ ]:
print('===== Training CNN =====')
hist_cnn = fit(cnn, epochs=3)

## 9. Compare the two models

Plot how test accuracy changes over epochs. You can see that the CNN usually reaches a
higher accuracy faster.


In [ ]:
ep = range(1, len(hist_mlp) + 1)
acc_mlp = [h[2] for h in hist_mlp]
acc_cnn = [h[2] for h in hist_cnn]

plt.figure(figsize=(7, 4))
plt.plot(ep, acc_mlp, 'o-', label='MLP')
plt.plot(ep, acc_cnn, 's-', label='CNN')
plt.xlabel('epoch'); plt.ylabel('Test accuracy')
plt.title('MLP vs CNN'); plt.legend(); plt.grid(alpha=0.3)
plt.xticks(list(ep))
plt.show()

print('MLP final test accuracy: %.4f' % acc_mlp[-1])
print('CNN final test accuracy: %.4f' % acc_cnn[-1])

## 10. Confusion matrix: which digits does the model confuse most?

In the confusion matrix, the entry at row `i`, column `j` is the number of samples whose
**true label is `i` but were predicted as `j`**. The diagonal holds the correct
predictions. Bright spots off the diagonal reveal which digits the model mixes up
(classic confused pairs: 4↔9, 3↔5, 7↔1).


In [ ]:
@torch.no_grad()
def predict_all(model, loader):
    model.eval()
    preds, trues = [], []
    for xb, yb in loader:
        logits = model(xb.to(device))
        preds.append(logits.argmax(1).cpu())
        trues.append(yb)
    return torch.cat(preds).numpy(), torch.cat(trues).numpy()


pred, true = predict_all(cnn, test_loader)

cm = np.zeros((10, 10), dtype=int)
for t, p in zip(true, pred):
    cm[t, p] += 1

plt.figure(figsize=(6.5, 5.5))
plt.imshow(cm, cmap='Blues')
plt.colorbar(label='Number of samples')
plt.xlabel('Predicted label'); plt.ylabel('True label')
plt.title('CNN confusion matrix (test set)')
plt.xticks(range(10)); plt.yticks(range(10))
for i in range(10):
    for j in range(10):
        if cm[i, j] > 0:
            plt.text(j, i, cm[i, j], ha='center', va='center',
                     color='white' if i == j else 'black', fontsize=7)
plt.tight_layout()
plt.show()

## 11. Inspect where the model goes wrong

Let's pick out and plot the samples the CNN predicts incorrectly. Many of these "errors"
are genuinely hard for a human to recognize at a glance — which is exactly the point:
MNIST is already largely solved by modern models, and what's left is mostly sloppily
written, borderline samples.


In [ ]:
wrong = np.where(pred != true)[0]
print('CNN got %d / %d test images wrong' % (len(wrong), len(true)))

fig, axes = plt.subplots(2, 8, figsize=(13, 3.6))
for ax, idx in zip(axes.ravel(), wrong[:16]):
    img, _ = test_set[idx]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title('true %d -> pred %d' % (true[idx], pred[idx]), fontsize=9, color='red')
    ax.axis('off')
fig.suptitle('Samples misclassified by the CNN')
plt.tight_layout()
plt.show()

## 12. Summary and next steps

**Recap of the whole pipeline:**

1. Image = matrix of numbers; classification = mapping that matrix to one of 10 classes.
2. `ToTensor + Normalize` for preprocessing, `DataLoader` for batching.
3. The **MLP** flattens the pixels and ignores spatial structure — baseline ~97–98%.
4. The **CNN** uses convolution to preserve and exploit the 2D structure, with fewer
   parameters yet higher accuracy, reaching ~99%.
5. Use the trio of **accuracy + confusion matrix + misclassified samples** to understand
   model performance.

**Things to try next:**

- Increase epochs, tune the learning rate, add more convolutional layers;
- Add **data augmentation** (random shifts/rotations via `torchvision.transforms`) to
  further improve generalization;
- Move to harder datasets: `FashionMNIST` (clothing), `CIFAR-10` (color objects);
- Save the trained model: `torch.save(cnn.state_dict(), 'cnn_mnist.pt')`.

> Tip for recording the video: run the entire notebook once first to cache the data and
> results, then run it section by section while explaining, to avoid waiting on
> downloads/training.
